In [3]:
# ---------------------------------------------------------
# 1. Load Data
# ---------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("group_vol&macrodata.csv")

# Ensure chronological order
df = df.sort_values("Date").reset_index(drop=True)


# ---------------------------------------------------------
# 2. Handle Missing Values
# ---------------------------------------------------------
# Forward fill macro features and crypto gaps
df = df.ffill()

# Drop any remaining NaNs at the beginning
df = df.dropna().reset_index(drop=True)


# ---------------------------------------------------------
# 3. Define Feature Groups (USING ACTUAL CSV COLUMN NAMES)
# ---------------------------------------------------------

# Asset log returns
asset_log_returns = [
    "GSPC_Log_Ret",
    "GLD_Log_Ret",
    "SLV_Log_Ret",
    "CL_Log_Ret",
    "META_Log_Ret",
    "TSLA_Log_Ret",
    "NVDA_Log_Ret",
    "AAPL_Log_Ret",
    "MSTR_Log_Ret"
]

# Asset variance features
asset_variance = [
    "GSPC_Variance",
    "GLD_Variance",
    "SLV_Variance",
    "CL_Variance",
    "META_Variance",
    "TSLA_Variance",
    "NVDA_Variance",
    "AAPL_Variance",
    "MSTR_Variance"
]

# Macro features
macro_features = [
    "VIX",
    "OVX",
    "GVZ",
    "VVIX",
    "SKEW",
    "MOVE",
    "fed_funds_rate",
    "hy_credit_spread",
    "tips_5y_real_yield",
    "yield_spread_10y2y",
    "XLK_logret",
    "XLE_logret",
    "QQQ_logret",
    "SOXX_logret",
    "DXY_logret",
    "BTC_logret",
    "meta_earnings",
    "tsla_earnings",
    "nvda_earnings",
    "aapl_earnings",
    "mstr_earnings"
]

# Combine all input features
feature_columns = asset_log_returns + asset_variance + macro_features


# ---------------------------------------------------------
# 4. Target Columns (9 variances)
# ---------------------------------------------------------
target_columns = [
    "GSPC_Target_Variance_t+1",
    "GLD_Target_Variance_t+1",
    "SLV_Target_Variance_t+1",
    "CL_Target_Variance_t+1",
    "META_Target_Variance_t+1",
    "TSLA_Target_Variance_t+1",
    "NVDA_Target_Variance_t+1",
    "AAPL_Target_Variance_t+1",
    "MSTR_Target_Variance_t+1"
]

X_raw = df[feature_columns].values
y_raw = df[target_columns].values


# ---------------------------------------------------------
# 5. Scaling
# ---------------------------------------------------------
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)


# ---------------------------------------------------------
# 6. 3D Tensor Formatting
# ---------------------------------------------------------
lookback = 21

X, y = [], []

for i in range(lookback, len(X_scaled)):

    # Past 21 days of features
    X.append(X_scaled[i-lookback:i])

    # Variance targets for day i
    y.append(y_scaled[i])

X = np.array(X)
y = np.array(y)


# ---------------------------------------------------------
# 7. Chronological Train/Test Split
# ---------------------------------------------------------
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


# ---------------------------------------------------------
# 8. Shape Check
# ---------------------------------------------------------
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2254, 21, 39)
y_train shape: (2254, 9)
